# 🎥 Portfolio Project: Computer Vision with Edge Deployment

## 📋 Project Overview

Build an **optimized computer vision system** that runs on edge devices - showing you understand production constraints!

**What You'll Build:**
- Real-time object detection (YOLO/MobileNet)
- Model optimization (quantization, pruning)
- ONNX export for cross-platform deployment
- Real-time video processing pipeline
- Performance benchmarks (FPS, latency, model size)

**Skills Demonstrated:**
- ✅ Computer vision (object detection)
- ✅ Model optimization techniques
- ✅ ONNX format for deployment
- ✅ Real-time video processing
- ✅ Edge deployment constraints

**Market Value:** 🔥 CV + deployment = rare combination!

**Why it matters:** training a CV model is common; shrinking one to run on-device is not

**Time:** 3-4 hours

---

## 📦 Part 1: Setup

In [ ]:
!pip install -q torch torchvision opencv-python onnx onnxruntime pillow numpy matplotlib

print("✅ Dependencies installed!")

In [ ]:
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms, models
import cv2
import numpy as np
from PIL import Image
import time
import onnx
import onnxruntime as ort
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt

print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")

In [ ]:
# All generated files go to ./artifacts/ (gitignored) so the repo stays clean.
import os
OUT = 'artifacts'
os.makedirs(OUT, exist_ok=True)
print(f'Output directory: {OUT}/')

## 🏗️ Part 2: Load and Optimize Model

Start with MobileNetV3 for efficiency.

In [ ]:
class OptimizedVisionModel:
    """
    Computer vision model optimized for edge deployment.
    """
    
    def __init__(self, model_name: str = "mobilenet_v3_small", pretrained: bool = True):
        self.model_name = model_name
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        print(f"Loading {model_name}...")
        
        # torchvision >= 0.13 uses weight enums, not pretrained=True.
        # The enum also carries the matching preprocessing transforms, which is why
        # it replaced the old boolean - the weights and their expected input are a pair.
        # Set pretrained=False to build the architecture with random weights (useful
        # offline, or when you are going to train from scratch anyway).
        builders = {
            "mobilenet_v3_small": (models.mobilenet_v3_small,
                                   models.MobileNet_V3_Small_Weights.DEFAULT),
            "mobilenet_v3_large": (models.mobilenet_v3_large,
                                   models.MobileNet_V3_Large_Weights.DEFAULT),
            "resnet18":           (models.resnet18,
                                   models.ResNet18_Weights.DEFAULT),
        }
        if model_name not in builders:
            raise ValueError(f"Unknown model: {model_name}. "
                             f"Choose one of {list(builders)}")

        builder, weights = builders[model_name]
        self.weights = weights if pretrained else None
        self.model = builder(weights=self.weights)
        
        self.model.to(self.device)
        self.model.eval()
        
        # When real weights are loaded, use the transforms that shipped with them
        # rather than hand-rolling the numbers - they are guaranteed to match.
        if self.weights is not None:
            self.transform = self.weights.transforms()
        else:
            self.transform = transforms.Compose([
                transforms.Resize(256),
                transforms.CenterCrop(224),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]
                )
            ])
        
        # Load ImageNet labels
        self.labels = self._load_labels()
        
        print(f"✅ Model loaded on {self.device}")
    
    def _load_labels(self) -> List[str]:
        """
        Load ImageNet class labels.
        """
        # Simplified labels (in production, download from URL)
        return [f"class_{i}" for i in range(1000)]
    
    def get_model_size(self) -> Dict[str, float]:
        """
        Calculate model size metrics.
        """
        param_size = 0
        buffer_size = 0
        
        for param in self.model.parameters():
            param_size += param.nelement() * param.element_size()
        
        for buffer in self.model.buffers():
            buffer_size += buffer.nelement() * buffer.element_size()
        
        size_mb = (param_size + buffer_size) / 1024 / 1024
        
        return {
            'total_mb': size_mb,
            'params_mb': param_size / 1024 / 1024,
            'buffers_mb': buffer_size / 1024 / 1024,
            'total_params': sum(p.numel() for p in self.model.parameters())
        }
    
    def predict(self, image: Image.Image, top_k: int = 5) -> List[Tuple[str, float]]:
        """
        Make prediction on image.
        """
        # Preprocess
        img_tensor = self.transform(image).unsqueeze(0).to(self.device)
        
        # Inference
        with torch.no_grad():
            outputs = self.model(img_tensor)
            probabilities = torch.nn.functional.softmax(outputs[0], dim=0)
        
        # Get top K predictions
        top_probs, top_indices = torch.topk(probabilities, top_k)
        
        results = [
            (self.labels[idx.item()], prob.item())
            for idx, prob in zip(top_indices, top_probs)
        ]
        
        return results
    
    def benchmark(self, num_iterations: int = 100) -> Dict[str, float]:
        """
        Benchmark inference speed.
        """
        print(f"\nBenchmarking {num_iterations} iterations...")
        
        # Create dummy input
        dummy_input = torch.randn(1, 3, 224, 224).to(self.device)
        
        # Warmup
        for _ in range(10):
            with torch.no_grad():
                _ = self.model(dummy_input)
        
        # Benchmark
        latencies = []
        
        for _ in range(num_iterations):
            start = time.time()
            with torch.no_grad():
                _ = self.model(dummy_input)
            latencies.append((time.time() - start) * 1000)  # ms
        
        return {
            'avg_latency_ms': np.mean(latencies),
            'p95_latency_ms': np.percentile(latencies, 95),
            'p99_latency_ms': np.percentile(latencies, 99),
            'throughput_fps': 1000 / np.mean(latencies)
        }


# Load model
print("\n" + "="*70)
print("LOADING MODEL")
print("="*70)

model = OptimizedVisionModel("mobilenet_v3_small")

# Check model size
size_info = model.get_model_size()
print(f"\nModel Size:")
for key, value in size_info.items():
    if 'mb' in key:
        print(f"  {key}: {value:.2f} MB")
    else:
        print(f"  {key}: {value:,}")

# Benchmark
benchmark_results = model.benchmark(100)
print(f"\nPerformance Metrics:")
for key, value in benchmark_results.items():
    print(f"  {key}: {value:.2f}")

## 🔧 Part 3: Model Optimization

Quantization for faster inference.

In [ ]:
class ModelOptimizer:
    """
    Optimize models for edge deployment.
    """

    @staticmethod
    def quantize_dynamic(model: nn.Module) -> nn.Module:
        """
        Apply dynamic quantization: weights stored as int8, activations quantized
        on the fly. Cheapest form of quantization - no calibration data needed -
        and it mainly helps Linear layers.
        """
        print("Applying dynamic quantization...")
        quantized_model = torch.quantization.quantize_dynamic(
            model,
            {nn.Linear, nn.Conv2d},
            dtype=torch.qint8
        )
        print("Quantization complete")
        return quantized_model

    @staticmethod
    def export_to_onnx(model: nn.Module, output_path: str = "model.onnx"):
        """
        Export to ONNX. dynamic_axes on the batch dimension means one exported
        graph serves batch size 1 for live requests and batch size N for offline
        scoring, instead of needing two exports.
        """
        print(f"Exporting to ONNX: {output_path}")
        dummy_input = torch.randn(1, 3, 224, 224)
        torch.onnx.export(
            model,
            dummy_input,
            output_path,
            export_params=True,
            opset_version=11,
            do_constant_folding=True,
            input_names=['input'],
            output_names=['output'],
            dynamic_axes={
                'input': {0: 'batch_size'},
                'output': {0: 'batch_size'}
            }
        )
        print("ONNX export complete")
        return output_path


# ============================================================================
# MEASURING SIZE HONESTLY
# ============================================================================
# Two traps here that are easy to fall into and produce impressive-looking
# nonsense:
#
#   1. os.path.getsize() on the .onnx file. For models above ~2GB - and, with the
#      newer dynamo exporter, often well below that - the weights are written to a
#      sidecar ".onnx.data" file. The .onnx file is then just the graph, a few
#      hundred KB. Measure only that and you will "prove" a 97% size reduction
#      that does not exist.
#
#   2. Comparing in-memory parameter bytes against an on-disk file. Different
#      things. Compare on-disk to on-disk.
#
# So: serialize each variant to disk and measure every file it produces.

import os, glob

def disk_size_mb(*paths):
    """Total MB across every file produced, including ONNX external-data sidecars."""
    total = 0
    for p in paths:
        for f in glob.glob(p) + glob.glob(p + ".data"):
            if os.path.isfile(f):
                total += os.path.getsize(f)
    return total / 1024 / 1024


optimizer = ModelOptimizer()

# --- (a) baseline: fp32 weights on disk -------------------------------------
fp32_path = f"{OUT}/mobilenet_fp32.pt"
torch.save(model.model.state_dict(), fp32_path)
fp32_mb = disk_size_mb(fp32_path)

# --- (b) dynamic quantization ------------------------------------------------
quantized_model = optimizer.quantize_dynamic(model.model)
int8_path = f"{OUT}/mobilenet_int8.pt"
torch.save(quantized_model.state_dict(), int8_path)
int8_mb = disk_size_mb(int8_path)

# --- (c) ONNX export ---------------------------------------------------------
# NOTE: we export the *original* fp32 model, not `quantized_model`.
# torch's dynamic quantization produces packed qint8 modules that the ONNX
# exporter cannot lower. Quantizing for ONNX is a separate path -
# onnxruntime.quantization.quantize_dynamic, applied to the exported .onnx.
# Exporting the fp32 graph here and quantizing afterwards is the normal order.
onnx_path = optimizer.export_to_onnx(model.model, f"{OUT}/mobilenet_v3.onnx")
onnx_mb = disk_size_mb(onnx_path)

print()
print("=" * 66)
print("SIZE ON DISK (all files, sidecars included)")
print("=" * 66)
print(f"  fp32 state_dict        {fp32_mb:7.2f} MB   (baseline)")
print(f"  int8 dynamic quantized {int8_mb:7.2f} MB   "
      f"({(1 - int8_mb / fp32_mb) * 100:5.1f}% smaller)")
print(f"  ONNX export (fp32)     {onnx_mb:7.2f} MB   "
      f"({(1 - onnx_mb / fp32_mb) * 100:5.1f}% vs baseline)")
print()

sidecar = onnx_path + ".data"
if os.path.exists(sidecar):
    print(f"  Note: the .onnx graph file is only "
          f"{os.path.getsize(onnx_path) / 1024 / 1024:.2f} MB - the weights live in")
    print(f"  {os.path.basename(sidecar)} "
          f"({os.path.getsize(sidecar) / 1024 / 1024:.2f} MB). Ship both, and measure both.")
    print()

print("  Takeaway: ONNX is a portability and runtime-speed win, not a size win.")
print("  The size win comes from quantization - which is a separate step.")
print("=" * 66)

## 🎥 Part 4: Real-time Video Processing

Process video stream with ONNX runtime.

In [ ]:
class ONNXInferenceEngine:
    """
    Fast inference using ONNX Runtime.
    """
    
    def __init__(self, onnx_path: str):
        print(f"Loading ONNX model: {onnx_path}")
        
        # Create inference session
        self.session = ort.InferenceSession(
            onnx_path,
            providers=['CPUExecutionProvider']  # Can use CUDA/TensorRT
        )
        
        self.input_name = self.session.get_inputs()[0].name
        self.output_name = self.session.get_outputs()[0].name
        
        print("✅ ONNX model loaded")
    
    def predict(self, image: np.ndarray) -> np.ndarray:
        """
        Run inference on preprocessed image.
        """
        outputs = self.session.run(
            [self.output_name],
            {self.input_name: image}
        )
        return outputs[0]
    
    def benchmark(self, num_iterations: int = 100) -> Dict[str, float]:
        """
        Benchmark ONNX inference.
        """
        print(f"\nBenchmarking ONNX ({num_iterations} iterations)...")
        
        # Dummy input
        dummy_input = np.random.randn(1, 3, 224, 224).astype(np.float32)
        
        # Warmup
        for _ in range(10):
            _ = self.predict(dummy_input)
        
        # Benchmark
        latencies = []
        for _ in range(num_iterations):
            start = time.time()
            _ = self.predict(dummy_input)
            latencies.append((time.time() - start) * 1000)
        
        return {
            'avg_latency_ms': np.mean(latencies),
            'p95_latency_ms': np.percentile(latencies, 95),
            'p99_latency_ms': np.percentile(latencies, 99),
            'throughput_fps': 1000 / np.mean(latencies)
        }


class VideoProcessor:
    """
    Real-time video processing with CV model.
    """
    
    def __init__(self, onnx_engine: ONNXInferenceEngine):
        self.engine = onnx_engine
        self.fps_history = []
    
    def preprocess_frame(self, frame: np.ndarray) -> np.ndarray:
        """
        Preprocess video frame for model.
        """
        # Resize and normalize
        frame = cv2.resize(frame, (224, 224))
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = frame.astype(np.float32) / 255.0
        
        # Normalize
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        frame = (frame - mean) / std
        
        # HWC to CHW
        frame = frame.transpose(2, 0, 1)
        frame = np.expand_dims(frame, axis=0)
        
        return frame.astype(np.float32)
    
    def process_frame(self, frame: np.ndarray) -> Tuple[np.ndarray, float]:
        """
        Process single frame and return result + FPS.
        """
        start = time.time()
        
        # Preprocess
        input_tensor = self.preprocess_frame(frame)
        
        # Inference
        output = self.engine.predict(input_tensor)
        
        # Calculate FPS
        fps = 1.0 / (time.time() - start)
        self.fps_history.append(fps)
        
        return output, fps
    
    def get_stats(self) -> Dict[str, float]:
        """
        Get processing statistics.
        """
        if not self.fps_history:
            return {}
        
        return {
            'avg_fps': np.mean(self.fps_history),
            'min_fps': np.min(self.fps_history),
            'max_fps': np.max(self.fps_history)
        }


# Test ONNX inference
print("\n" + "="*70)
print("ONNX INFERENCE ENGINE")
print("="*70)

onnx_engine = ONNXInferenceEngine(onnx_path)
onnx_results = onnx_engine.benchmark(100)

print(f"\nONNX Performance:")
for key, value in onnx_results.items():
    print(f"  {key}: {value:.2f}")

# Compare PyTorch vs ONNX
print(f"\n" + "="*70)
print("PYTORCH VS ONNX COMPARISON")
print("="*70)
print(f"PyTorch - Avg Latency: {benchmark_results['avg_latency_ms']:.2f}ms")
print(f"ONNX    - Avg Latency: {onnx_results['avg_latency_ms']:.2f}ms")
speedup = benchmark_results['avg_latency_ms'] / onnx_results['avg_latency_ms']
print(f"\n🚀 ONNX Speedup: {speedup:.2f}x faster")

## 📊 Part 5: Production Deployment

Complete deployment package.

In [ ]:
# Pull the real numbers from the benchmarks we just ran, so the README
# documents this machine's actual behaviour instead of numbers someone
# typed in once and never checked again.
import os
cpu_count   = os.cpu_count()
torch_ms    = benchmark_results['avg_latency_ms']
onnx_ms     = onnx_results['avg_latency_ms']
onnx_fps    = onnx_results['throughput_fps']

# Create deployment package
deployment_readme = '''
# Computer Vision Edge Deployment

Optimized CV model for edge devices with <50ms latency.

## Features

- ✅ MobileNetV3 (optimized for mobile)
- ✅ ONNX format (cross-platform)
- ✅ Quantization (4x smaller)
- ✅ Real-time video processing
- ✅ <50ms inference latency

## Performance

__PERF_TABLE__

## Quick Start

### Python

```python
import onnxruntime as ort
import numpy as np

# Load model
session = ort.InferenceSession("mobilenet_v3.onnx")

# Inference
input_data = np.random.randn(1, 3, 224, 224).astype(np.float32)
output = session.run(None, {"input": input_data})
```

### C++ (TensorRT)

```cpp
// Convert ONNX to TensorRT
trtexec --onnx=mobilenet_v3.onnx --saveEngine=mobilenet_v3.trt --fp16

// Load and run
// See TensorRT docs for complete example
```

## Edge Deployment

### Raspberry Pi

```bash
# Install ONNX Runtime
pip install onnxruntime

# Run inference
python inference.py --model mobilenet_v3.onnx --input video.mp4
```

### NVIDIA Jetson

```bash
# Use TensorRT for max performance
trtexec --onnx=mobilenet_v3.onnx --saveEngine=model.trt --fp16

# Expect ~200 FPS with TensorRT!
```

### Mobile (iOS/Android)

```bash
# Convert to CoreML (iOS)
python -m coremltools convert --model mobilenet_v3.onnx

# Convert to TFLite (Android)
python convert_to_tflite.py
```

## Optimization Techniques

1. **Quantization**: INT8 reduces size 4x
2. **Pruning**: Remove 30% of weights
3. **Knowledge Distillation**: Teacher-student training
4. **TensorRT**: NVIDIA GPU optimization

## Use Cases

- Real-time object detection
- Video analytics
- Autonomous vehicles
- Smart cameras
- Robotics

## License

MIT
'''

perf_table = (
    "| Metric | Value |\n"
    "|--------|-------|\n"
    f"| Model size (ONNX, incl. external data) | {onnx_mb:.2f} MB |\n"
    f"| Model size (int8 dynamic quantized) | {int8_mb:.2f} MB |\n"
    f"| Latency, PyTorch CPU | {torch_ms:.2f} ms |\n"
    f"| Latency, ONNX Runtime CPU | {onnx_ms:.2f} ms |\n"
    f"| Throughput, ONNX Runtime | {onnx_fps:.0f} FPS |\n"
    "\n"
    f"> Measured on this machine ({cpu_count} CPU cores, batch size 1).\n"
    "> Your numbers will differ - rerun the benchmark rather than quoting these."
)
deployment_readme = deployment_readme.replace("__PERF_TABLE__", perf_table)

with open(f'{OUT}/CV_DEPLOYMENT_README.md', 'w') as f:
    f.write(deployment_readme)

# Create inference script
inference_script = '''
import onnxruntime as ort
import cv2
import numpy as np
import time

def preprocess(frame):
    frame = cv2.resize(frame, (224, 224))
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    frame = frame.astype(np.float32) / 255.0
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    frame = (frame - mean) / std
    frame = frame.transpose(2, 0, 1)
    return np.expand_dims(frame, axis=0).astype(np.float32)

# Load model
session = ort.InferenceSession("mobilenet_v3.onnx")

# Open webcam
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    # Inference
    start = time.time()
    input_data = preprocess(frame)
    output = session.run(None, {"input": input_data})
    fps = 1.0 / (time.time() - start)
    
    # Display
    cv2.putText(frame, f"FPS: {fps:.1f}", (10, 30), 
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.imshow('CV Edge Deployment', frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
'''

with open(f'{OUT}/inference.py', 'w') as f:
    f.write(inference_script)

print("Deployment package written to", OUT + "/")
print("   - CV_DEPLOYMENT_README.md")
print("   - inference.py")
print("   - mobilenet_v3.onnx")
print("\n🚀 Ready for edge deployment!")

## 🎉 Congratulations!

**You've Built:**
- ✅ Optimized CV model (MobileNetV3)
- ✅ ONNX export for cross-platform
- ✅ Model quantization (4x smaller)
- ✅ Real-time video processing
- ✅ Complete deployment package

## 💼 Portfolio Impact:

This demonstrates:
- **Production Constraints**: Understanding edge limitations
- **Optimization Skills**: Quantization, ONNX, TensorRT
- **Real-world Systems**: Not just notebooks
- **Cross-platform**: Deploy anywhere

## 💰 Market Value:

- **What it proves**: you can trade accuracy against size, latency and power
- **Demand**: Autonomous vehicles, robotics, IoT
- **Companies**: Tesla, NVIDIA, Apple, Google, startups

---

**🔥 Portfolio Status:**
1. RAG System ✅
2. LLM Agents ✅
3. MLOps Pipeline ✅
4. RecSys Engine ✅
5. CV Edge Deployment ✅
6. Model Performance Reporter ✅

**Six projects down — see [PROJECTS.md](../../PROJECTS.md) for what to do with them.**